In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# 1. Cargar el dataset crudo (Garantiza portabilidad absoluta)
df = pd.read_csv("../01.data/raw/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remover ID para evitar que provoque el ValueError
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

X = df.drop("Churn", axis=1)
y = df["Churn"].map({"Yes": 1, "No": 0})

# 2. Segmentar columnas automáticamente
num_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

# 3. División balanceada (Stratify evita sesgos en el Churn)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Creación del Pipeline para la Regresión Logística
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# Entrenar el modelo base de manera segura
lr_pipeline.fit(X_train, y_train)
print("✅ ¡Pipeline de Regresión Logística entrenado con éxito sin errores de conversión!")

# 5. Evaluar desempeño del modelo base
y_pred = lr_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print("🧩 MATRIZ DE CONFUSIÓN:")
print(confusion_matrix(y_test, y_pred))

✅ ¡Pipeline de Regresión Logística entrenado con éxito sin errores de conversión!
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

🧩 MATRIZ DE CONFUSIÓN:
[[926 109]
 [165 209]]


In [23]:
#6. Guardar resultados y modelo
import joblib
joblib.dump(model, "../01.models/logistic_regression_telco.pkl")
print("Modelo guardado correctamente.")

Modelo guardado correctamente.


In [24]:
#7. Importar el modelo
from sklearn.ensemble import RandomForestClassifier

In [25]:
#8. Entrenar el modelo (Random Forest)
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# 1. Definimos el Pipeline usando el 'preprocessor' que ya creamos en el paso de Regresión Logística
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor), # Reutiliza el transformador que creamos antes
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

# 2. Entrenar el Pipeline de Random Forest de forma segura
print("🌲 Entrenando Random Forest...")
rf_pipeline.fit(X_train, y_train)
print("✅ ¡Random Forest entrenado con éxito sin errores de conversión!")

# 3. Evaluar el modelo
from sklearn.metrics import classification_report
y_pred_rf = rf_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_rf))
print("🧩 MATRIZ DE CONFUSIÓN:")
print(confusion_matrix(y_test, y_pred_rf))

🌲 Entrenando Random Forest...
✅ ¡Random Forest entrenado con éxito sin errores de conversión!
              precision    recall  f1-score   support

           0       0.82      0.89      0.86      1035
           1       0.61      0.47      0.53       374

    accuracy                           0.78      1409
   macro avg       0.71      0.68      0.69      1409
weighted avg       0.77      0.78      0.77      1409

🧩 MATRIZ DE CONFUSIÓN:
[[920 115]
 [197 177]]


In [26]:
#10. Guardar el modelo
import joblib
joblib.dump(rf, "../01.models/random_forest_telco.pkl")
print("Modelo Random Forest guardado.")

Modelo Random Forest guardado.


In [27]:
#11. Instalar XGBoost
!pip install xgboost

In [28]:
#12. Importar XGBoost
from xgboost import XGBClassifier

In [31]:
#13. Entrenar el modelo (XGBoost)
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Envolvemos a XGBoost en el Pipeline usando el transformador que creamos antes
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor), # Esto convierte de 'str' a flotantes numéricos automáticamente
    ("model", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        
        # --- POTENCIA DE TU HARDWARE: ACTIVACIÓN DE TU RTX 4060 ---
        tree_method="hist",
        device="cuda"
    ))
])

# 2. Entrenamos el modelo de forma acelerada por hardware
print("🚀 Entrenando XGBoost en tu GPU NVIDIA RTX 4060 dentro de Docker...")
xgb_pipeline.fit(X_train, y_train)
print("✅ ¡XGBoost entrenado con éxito sin errores de tipos de datos!")

# 3. Evaluamos su rendimiento con el set de pruebas
y_pred_xgb = xgb_pipeline.predict(X_test)
print("\n📊 REPORTE DE RENDIMIENTO DE XGBOOST:")
print(classification_report(y_test, y_pred_xgb))
print("🧩 MATRIZ DE CONFUSIÓN:")
print(confusion_matrix(y_test, y_pred_xgb))

🚀 Entrenando XGBoost en tu GPU NVIDIA RTX 4060 dentro de Docker...
✅ ¡XGBoost entrenado con éxito sin errores de tipos de datos!

📊 REPORTE DE RENDIMIENTO DE XGBOOST:
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1035
           1       0.62      0.53      0.57       374

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.78      0.79      0.78      1409

🧩 MATRIZ DE CONFUSIÓN:
[[913 122]
 [174 200]]


In [32]:
#15. Guardar el modelo
import joblib
joblib.dump(xgb, "../01.models/xgboost_telco.pkl")
print("Modelo XGBoost guardado.")

Modelo XGBoost guardado.


In [33]:
#16. Crear función para evaluar modelos
from os import name

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(name, y_true, y_pred):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }

In [34]:
#17. Evaluar cada modelo
results = []

# Logistic Regression
results.append(evaluate_model("Logistic Regression", y_test,
y_pred))

# Random Forest
results.append(evaluate_model("Random Forest", y_test, y_pred_rf))

# XGBoost
results.append(evaluate_model("XGBoost", y_test, y_pred_xgb))

In [35]:
#18. Mostrar tabla comparativa
import pandas as pd

df_results = pd.DataFrame(results)
df_results.sort_values(by="f1", ascending=False)

,model,accuracy,precision,recall,f1
0,Logistic Regression,0.805536,0.657233,0.558824,0.604046
2,XGBoost,0.789922,0.621118,0.534759,0.574713
1,Random Forest,0.778566,0.606164,0.473262,0.531532


In [36]:
#19. Pipeline completo Telco Churn (preprocesamiento + XGBoost)
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

# 1. Cargar dataset crudo original para el pipeline reproducible
df = pd.read_csv("../01.data/raw/telco_churn.csv")

# 2. Preparación inicial limpia (Evita el ChainedAssignmentError)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.drop(columns=["customerID"])

# 3. Separación de variables predictoras y objetivo
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"Yes": 1, "No": 0})

# 4. Segmentación estricta de columnas
num_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]

# 5. Dividir los datos ANTES de procesar para evitar fuga de información (Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 6. DEFINICIÓN DE TRANSFORMADORES INTERNOS DEL PIPELINE
# Transformador numérico: Imputa la mediana y escala los datos (Soluciona la convergencia)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Transformador categórico: One-Hot Encoding seguro
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Unificador del Preprocesamiento
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

# 7. CONSTRUCCIÓN DEL PIPELINE ACELERADO POR GPU (RTX 4060)
pipeline_produccion = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        
        # --- ACTIVACIÓN DE TU GPU NVIDIA EN DOCKER ---
        tree_method="hist",
        device="cuda"
    ))
])

# 8. Entrenar todo el flujo de forma masiva y veloz
print("🚀 Entrenando XGBoost en tu GPU NVIDIA RTX 4060 dentro de Docker...")
pipeline_produccion.fit(X_train, y_train)

# 9. Guardar el artefacto de producción final
joblib.dump(pipeline_produccion, "../01.models/pipeline_xgboost_telco.pkl")
print("✅ Pipeline integrado de producción exportado con éxito.")

🚀 Entrenando XGBoost en tu GPU NVIDIA RTX 4060 dentro de Docker...
✅ Pipeline integrado de producción exportado con éxito.


In [37]:
#20. Validación y Evaluación del pipeline completo
# Evaluar el comportamiento del nuevo pipeline balanceado
y_pred = pipeline_produccion.predict(X_test)

print("\n📊 REPORTE DE CLASIFICACIÓN EN PRODUCCIÓN:")
print(classification_report(y_test, y_pred))

print("🧩 MATRIZ DE CONFUSIÓN:")
print(confusion_matrix(y_test, y_pred))


📊 REPORTE DE CLASIFICACIÓN EN PRODUCCIÓN:
              precision    recall  f1-score   support

           0       0.84      0.89      0.87      1035
           1       0.65      0.55      0.59       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.79      0.80      0.79      1409

🧩 MATRIZ DE CONFUSIÓN:
[[923 112]
 [170 204]]
